#  Distorted Captcha Visual Sequence Recognition (CRNN + CTC Loss)

This notebook contains a complete end-to-end production pipeline for recognizing sequence patterns in visually distorted alphanumeric grayscale images. The pipeline leverages a Deep Learning **Convolutional Recurrent Neural Network (CRNN)** paired with a **Connectionist Temporal Classification (CTC)** loss function to handle unaligned character sequences.

###  Section 1:
1. **Environment Setup:** Imports necessary deep learning frameworks (`PyTorch`, `Torchvision`) and data processing libraries (`pandas`, `PIL`).
2. **Global Vocabulary Config:** Defines the 36 alphanumeric character vocabulary and handles two-way numerical mappings.
3. **Image Transformations:** Resizes images to a unified profile ($64 \times 200$), converts them to single-channel grayscale, and applies global channel normalization.
4. **Self-Healing Dataset Class (`CaptchaCSVDataset`):** An intelligent PyTorch Dataset implementation that dynamically auto-detects column targets, cleans trailing whitespaces, fixes missing file extensions, and builds label vectors.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import torchvision.transforms as transforms

# 1. Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Character Vocabulary
CHAR_LIST = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
char_to_num = {char: idx + 1 for idx, char in enumerate(CHAR_LIST)}
num_to_char = {idx + 1: char for idx, char in enumerate(CHAR_LIST)}
num_classes = len(CHAR_LIST) + 1

# 3. Image Transformations
img_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# 4. Smart Self-Healing Dataset Class
class CaptchaCSVDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        
        img_cols = [c for c in self.df.columns if 'img' in str(c).lower() or 'image' in str(c).lower()]
        label_cols = [c for c in self.df.columns if 'label' in str(c).lower() or 'text' in str(c).lower()]
        
        self.img_col = img_cols[0] if img_cols else self.df.columns[0]
        self.label_col = label_cols[0] if label_cols else self.df.columns[1]
        
        print(f"🔍 Auto-detected columns -> Image Name Column: '{self.img_col}' | Label Column: '{self.label_col}'")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = str(self.df.iloc[idx][self.img_col]).strip()
        if not img_name.lower().endswith('.png'):
            img_name += '.png'
            
        label_text = str(self.df.iloc[idx][self.label_col]).upper().strip()
        
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path)
        
        if self.transform:
            image = self.transform(image)
            
        label_tensor = torch.tensor([char_to_num[c] for c in label_text if c in char_to_num], dtype=torch.long)
        return image, label_tensor, torch.tensor(len(label_tensor), dtype=torch.long), img_name

def ctc_collate_fn(batch):
    images, labels, label_lengths, img_names = zip(*batch)
    images = torch.stack(images, 0)
    flat_labels = torch.cat(labels, dim=0)
    label_lengths = torch.stack(label_lengths, 0)
    return images, flat_labels, label_lengths, img_names

##  Section 2: Data Splitting & High-Throughput DataLoaders

This block manages dataset isolation and constructs performance-optimized batch generation engines.

* **Deterministic Splitting:** Splitting the comprehensive dataset into a 90% Training pool and a 10% Validation pool using a static manual seed to ensure exact cross-session consistency.
* **Maximized Processing Throughput:** Uses an aggressive batch size of `256` wrapped with memory pinning (`pin_memory=True`) to accelerate data delivery to your target hardware acceleration device (CUDA GPU).
* **Collate Customization (`ctc_collate_fn`):** Unpacks and rearranges multi-length character arrays into compressed continuous sequence tensors required directly by PyTorch's CTC Loss Engine.

In [ ]:
CSV_PATH = r"C:\Users\aadit\Downloads\cig_ps\cig_ps\train-labels.csv"
IMG_DIR = r"C:\Users\aadit\Downloads\cig_ps\cig_ps\train_images"

full_dataset = CaptchaCSVDataset(csv_file=CSV_PATH, img_dir=IMG_DIR, transform=img_transforms)

train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# Using the optimized batch size of 256 immediately
BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=ctc_collate_fn, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=ctc_collate_fn, num_workers=0, pin_memory=True)

print(f" Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")
print(f" Data loaders maximized! Processing batches of {BATCH_SIZE} images.")

🔍 Auto-detected columns -> Image Name Column: 'image' | Label Column: 'text'
📊 Train samples: 18000 | Val samples: 2000
⚡ Data loaders maximized! Processing batches of 256 images.


##  Section 3: Deep CRNN Model Architecture (Anti-Collapse Framework)

The model features an advanced hybrid design combining deep spatial feature extraction with context-aware sequential pattern prediction.

###  Architectural Layers:
1. **CNN Feature Extractor:** Composed of 5 progressive convolutional blocks. It implements `BatchNorm2d` for feature scale preservation, `LeakyReLU` activations to completely prevent "dying gradients," and custom rectangular asymmetric pooling kernels (`MaxPool2d`) to protect valuable horizontal width context for deep sequence recognition.
2. **Spatial Feature Compression:** Squeezes and permutes the multidimensional visual tensors into sequential feature strings.
3. **Bidirectional LSTM Core:** Employs a stacked two-layer bidirectional recurrent engine with a `0.3` recurrent dropout floor to avoid pattern loop traps and character repetitions.
4. **Classification Output Projection:** A linear mapping layer transforming sequential text representations into vocabulary probability logits over a 50-step time horizon.

In [ ]:
class CustomCaptchaCRNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCaptchaCRNN, self).__init__()
        
        # 1. CNN Feature Extractor with LeakyReLU to prevent dying gradients
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)), 
            
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout2d(0.2), # Randomly breaks local pattern reliance
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),
            
            nn.Conv2d(512, 512, kernel_size=(4, 1), stride=1, padding=0),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1, inplace=True)
        )
        
        # 2. Recurrent RNN Layers with Dropout to stop repetition loops
        self.rnn_dropout = nn.Dropout(0.3) 
        self.rnn1 = nn.LSTM(input_size=512, hidden_size=256, bidirectional=True, batch_first=True)
        self.rnn2 = nn.LSTM(input_size=512, hidden_size=256, bidirectional=True, batch_first=True)
        
        # 3. Dense Classification Output Layer
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        features = self.cnn(x)
        features = features.squeeze(2)        # [Batch, 512, 50]
        features = features.permute(0, 2, 1)  # [Batch, 50, 512]
        
        features = self.rnn_dropout(features)
        
        rnn_out, _ = self.rnn1(features)
        rnn_out, _ = self.rnn2(rnn_out)
        
        logits = self.fc(rnn_out)             # [Batch, 50, num_classes]
        return logits.permute(1, 0, 2)        # [50, Batch, num_classes]

# Instantiate the architecture to your target device
model = CustomCaptchaCRNN(num_classes=num_classes).to(device)
print(" Anti-collapse model architecture loaded and instantiated!")

⚔️ Anti-collapse model architecture loaded and instantiated!


##  Section 4: Tensor Shape & Flow Validation (Sanity Check)

Before launching the intensive training pipeline, this step extracts a single micro-batch from our streaming dataset loader and conducts a dummy execution pass through the CRNN. This guarantees that internal feature matrix multiplications, channel transformations, and temporal sequence dimensions perfectly align with structural specifications without raising hardware layer faults.

In [4]:
test_images, _, _, _ = next(iter(train_loader))
print(f"Input batch shape: {test_images.shape}")

with torch.no_grad():
    test_output = model(test_images.to(device))

print(f"Output prediction shape: {test_output.shape}")

Input batch shape: torch.Size([256, 1, 64, 200])
Output prediction shape: torch.Size([50, 256, 37])


##  Section 5: CTC Logit Decoding & Character Error Rate (CER) Evaluation

Because CTC loss calculates alignment probabilities over a dense step sequence, output logits require professional post-processing.

* **Greedy Path Decoder (`decode_ctc_predictions`):** Selects character indices with top-tier confidence metrics at each time slice, strips blank tokens, collapses duplicate character sequences, and translates the raw arrays back into readable text strings.
* **Levenshtein Distance Metric (`compute_levenshtein_distance`):** Employs a dynamic programming matrix algorithm to calculate minimum string edit edits (Insertions, Deletions, and Substitutions) between actual words and model interpretations.
* **Character Error Rate Engine (`calculate_batch_cer`):** Computes precise error frequencies globally across evaluation blocks to provide an absolute objective standard for accuracy tracking.

In [ ]:
import numpy as np

def decode_ctc_predictions(logits, num_to_char):
    preds = torch.argmax(logits, dim=2) 
    preds = preds.permute(1, 0)         
    
    decoded_texts = []
    for i in range(preds.size(0)):
        char_indices = preds[i].tolist()
        collapsed_text = []
        prev_idx = None
        
        for idx in char_indices:
            if idx != 0 and idx != prev_idx: 
                collapsed_text.append(num_to_char[idx])
            prev_idx = idx
            
        decoded_texts.append("".join(collapsed_text))
    return decoded_texts

def compute_levenshtein_distance(string1, string2):
    if len(string1) < len(string2):
        return compute_levenshtein_distance(string2, string1)
    if len(string2) == 0:
        return len(string1)
    
    prev_row = range(len(string2) + 1)
    for i, c1 in enumerate(string1):
        curr_row = [i + 1]
        for j, c2 in enumerate(string2):
            insertions = prev_row[j + 1] + 1
            deletions = curr_row[j] + 1
            substitutions = prev_row[j] + (c1 != c2)
            curr_row.append(min(insertions, deletions, substitutions))
        prev_row = curr_row
        
    return prev_row[-1]

def calculate_batch_cer(pred_strings, true_strings):
    total_distance = 0
    total_characters = 0
    for pred, true in zip(pred_strings, true_strings):
        total_distance += compute_levenshtein_distance(pred, true)
        total_characters += max(len(true), 1)
    return total_distance, total_characters

print(" CTC Decoding Engine & CER metrics compiled successfully!")

🎯 CTC Decoding Engine & CER metrics compiled successfully!


## 📈 Section 6: Real-Time Visualization Framework (TensorBoard)

Launches the local TensorBoard monitoring dashboard extension directly inside our interactive Jupyter panel. This connects live data tracking nodes to capture validation steps, loss curves, and evaluation metrics visually as training proceeds.

In [6]:
%load_ext tensorboard
%tensorboard --logdir runs

Reusing TensorBoard on port 6006 (pid 53156), started 7:43:23 ago. (Use '!kill 53156' to kill it.)

##  Section 7: Structural State Checkpoint Restoration

Retrieves and maps pre-compiled structural model weights (`best_captcha_crnn.pth`) directly back onto our live device-bound architecture instance. This allows the framework to either resume learning seamlessly from its past optimized state or instantly spin up validation engines without retraining from scratch.

In [ ]:
model.load_state_dict(torch.load("best_captcha_crnn.pth", map_location=device))
print(" Successfully loaded current best weights!")

💎 Successfully loaded current best weights!


##  Section 8: High-Performance AMP Training & Validation Loop

This cell runs the primary engine execution cycle. It drives model learning across a 50-epoch horizon while implementing state-of-the-art performance enhancements.

###  Optimization Implementations:
* **Automatic Mixed Precision (AMP):** Wraps operations inside `torch.cuda.amp.autocast()` and leverages a `GradScaler`. By dynamically shifting math operations between full Float32 and lightweight Float16 representation, it slashes overall VRAM requirements, accelerates compute throughput, and prevents underflow gradient collapses.
* **AdamW Optimization:** Applies advanced weight decay mechanics alongside an adaptive learning rate strategy to keep parameters tightly generalized.
* **Dynamic Learning Rate Plateau Scheduler:** Monitors validation data behavior and steps down step coefficients automatically if model convergence shows signs of slowing.
* **Progress Visualization:** Employs rich `tqdm` status bars to display live training cross-metrics continuously on the terminal screen.

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm
import time

# 1. Initialize structural TensorBoard logs
writer = SummaryWriter(log_dir="runs/captcha_crnn_optimization")

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# 2. PyTorch AMP Scaler for 16-bit execution acceleration
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

EPOCHS = 50
best_val_cer = float('inf')

print("🏎️ Speed-Optimized Engine Ignited! Check your TensorBoard panel above for live charts.")

for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.time()
    
    # ================= TRAINING PHASE =================
    model.train()
    running_train_loss = 0.0
    
    # Real-time visual progress bar tracking
    train_bar = tqdm(train_loader, desc=f"🎬 Epoch {epoch}/{EPOCHS} [Train]", leave=False)
    
    for images, flat_labels, label_lengths, _ in train_bar:
        images = images.to(device)
        batch_size = images.size(0)
        
        optimizer.zero_grad()
        
        # Runs forward steps inside accelerated low-precision environment
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(images)
            log_probs = logits.log_softmax(2)
            input_lengths = torch.full(size=(batch_size,), fill_value=50, dtype=torch.long, device=device)
            loss = criterion(log_probs, flat_labels.to(device), input_lengths, label_lengths.to(device))
        
        # Stabilized gradient backward tracking
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        scaler.step(optimizer)
        scaler.update()
        
        running_train_loss += loss.item() * batch_size
        train_bar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})
        
    avg_train_loss = running_train_loss / len(train_loader.dataset)
    
    # ================= VALIDATION PHASE =================
    model.eval()
    total_val_dist = 0
    total_val_chars = 0
    
    val_bar = tqdm(val_loader, desc=f"🔍 Epoch {epoch}/{EPOCHS} [Val]", leave=False)
    
    with torch.no_grad():
        for images, flat_labels, label_lengths, _ in val_bar:
            images = images.to(device)
            
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(images)
                
            pred_strings = decode_ctc_predictions(logits, num_to_char)
            
            true_strings = []
            current_idx = 0
            for length in label_lengths:
                label_slice = flat_labels[current_idx : current_idx + length.item()].tolist()
                true_str = "".join([num_to_char[num] for num in label_slice if num in num_to_char])
                true_strings.append(true_str)
                current_idx += length.item()
            
            dist, chars = calculate_batch_cer(pred_strings, true_strings)
            total_val_dist += dist
            total_val_chars += chars
            
    epoch_val_cer = total_val_dist / total_val_chars
    scheduler.step(epoch_val_cer)
    
    # 3. Push telemetry down to TensorBoard panels
    writer.add_scalar("Loss/Train", avg_train_loss, epoch)
    writer.add_scalar("Metric/Val_CER", epoch_val_cer, epoch)
    writer.add_scalar("System/Learning_Rate", optimizer.param_groups[0]['lr'], epoch)
    
    elapsed_time = time.time() - epoch_start_time
    print(f" Epoch [{epoch}/{EPOCHS}] ({elapsed_time:.1f}s) | Train Loss: {avg_train_loss:.4f} | Val CER: {epoch_val_cer:.4f}")
    
    if epoch_val_cer < best_val_cer:
        best_val_cer = epoch_val_cer
        torch.save(model.state_dict(), "best_captcha_crnn.pth")
        print(f" Checkpoint saved! New Floor: {best_val_cer:.4f}")

writer.close()
print(f"\n🎉 Optimization complete! Ultimate Validation CER: {best_val_cer:.4f}")

🏎️ Speed-Optimized Engine Ignited! Check your TensorBoard panel above for live charts.


🎬 Epoch 1/50 [Train]:   0%|          | 0/71 [00:00<?, ?it/s]

🔍 Epoch 1/50 [Val]:   0%|          | 0/8 [00:00<?, ?it/s]

🏅 Epoch [1/50] (60.4s) | Train Loss: 0.0317 | Val CER: 0.0070
💾 Checkpoint saved! New Floor: 0.0070


🎬 Epoch 2/50 [Train]:   0%|          | 0/71 [00:00<?, ?it/s]

...

🔍 Epoch 49/50 [Val]:   0%|          | 0/8 [00:00<?, ?it/s]

🏅 Epoch [49/50] (40.5s) | Train Loss: 0.0004 | Val CER: 0.0013


🎬 Epoch 50/50 [Train]:   0%|          | 0/71 [00:00<?, ?it/s]

🔍 Epoch 50/50 [Val]:   0%|          | 0/8 [00:00<?, ?it/s]

🏅 Epoch [50/50] (40.5s) | Train Loss: 0.0003 | Val CER: 0.0012

🎉 Optimization complete! Ultimate Validation CER: 0.0011


In [ ]:
model.load_state_dict(torch.load("best_captcha_crnn.pth", map_location=device))
print(" Whew! Best weights successfully re-loaded into memory.")

💎 Whew! Best weights successfully re-loaded into memory.


##  Section 9: Model Accuracy Verification & Visual Performance Dashboard

This cell serves as the final quality-assurance gateway for our pipeline. It provides direct, undeniable visual proof of the network's predictive capabilities before deployment or final file submission.

###  What this validation block performs:
1. **Pristine Weight Recall:** Explicitly targets and re-loads our absolute best-performing checkpoint file (`best_captcha_crnn.pth`) to guarantee evaluation metrics reflect peak network intelligence.
2. **Isolated Inference Execution:** Shifts the CRNN into `model.eval()` mode and wraps tensor calculations within `torch.no_grad()`. This deactivates dropout layers and freezes gradient tracking to preserve system memory and maximize computation velocity.
3. **Side-by-Side Verification Logs:** Pulls a random batch from our unseen validation loader, translates the raw spatial logits back into textual strings, and prints an explicit string comparison index marking each output as a complete match (`✅ MATCH`) or a discrepancy (`❌ MISMATCH`).
4. **Visual Sample Gallery:** Leverages `matplotlib` to denormalize and reconstruct 4 raw grayscale image tensors into visual subplots, printing the ground truth side-by-side with the model's live text prediction directly above the target CAPTCHA image.

In [ ]:
import matplotlib.pyplot as plt

# 1. Load the ultimate optimized weights
if os.path.exists("best_captcha_crnn.pth"):
    model.load_state_dict(torch.load("best_captcha_crnn.pth", map_location=device))
    print(" Loaded the best performing model weights successfully!")
else:
    print(" 'best_captcha_crnn.pth' not found. Using current in-memory model weights.")

model.eval()

# 2. Grab a single random batch from the validation loader
images, flat_labels, label_lengths, img_names = next(iter(val_loader))

# 3. Run predictions
with torch.no_grad():
    with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
        logits = model(images.to(device))
    pred_strings = decode_ctc_predictions(logits, num_to_char)

# 4. Reconstruct the true target strings from the flat batch data
true_strings = []
current_idx = 0
for length in label_lengths:
    label_slice = flat_labels[current_idx : current_idx + length.item()].tolist()
    true_str = "".join([num_to_char[num] for num in label_slice if num in num_to_char])
    true_strings.append(true_str)
    current_idx += length.item()

# 5. Print a clear text comparison dashboard for the first 15 samples
print("\n📋 SIDE-BY-SIDE EVALUATION SAMPLES:")
print(f"{'Image File':<25} | {'Ground Truth':<15} | {'Model Prediction':<15} | {'Status'}")
print("-" * 75)

for i in range(min(15, len(images))):
    status = "✅ MATCH" if true_strings[i] == pred_strings[i] else "❌ MISMATCH"
    print(f"{img_names[i]:<25} | {true_strings[i]:<15} | {pred_strings[i]:<15} | {status}")

# 6. Plot a visual gallery of 4 random samples using matplotlib
plt.figure(figsize=(12, 6))
for i in range(4):
    plt.subplot(2, 2, i + 1)
    
    # Denormalize image back to standard grayscale range for correct display
    img = images[i].squeeze().cpu().numpy()
    img = (img * 0.5) + 0.5  # Reverses Normalize(mean=[0.5], std=[0.5])
    
    plt.imshow(img, cmap='gray')
    
    color = 'green' if true_strings[i] == pred_strings[i] else 'red'
    plt.title(f"True: {true_strings[i]} | Pred: {pred_strings[i]}", color=color, fontsize=12, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()

💎 Loaded the best performing model weights successfully!

📋 SIDE-BY-SIDE EVALUATION SAMPLES:
Image File                | Ground Truth    | Model Prediction | Status
---------------------------------------------------------------------------
train-2257.png            | EUYYJ5          | EUYYJ5          | ✅ MATCH
train-6174.png            | W7D56Y          | W7D56Y          | ✅ MATCH
train-9143.png            | 3B2DYN          | 3B2DYN          | ✅ MATCH
train-7887.png            | TM9YQH          | TM9YQH          | ✅ MATCH
train-19328.png           | MSZMVM          | MSZMVM          | ✅ MATCH
train-10955.png           | X329YU          | X329YU          | ✅ MATCH
train-18526.png           | PXQACV          | PXQACV          | ✅ MATCH
train-12235.png           | 6439VX          | 6439VX          | ✅ MATCH
train-9247.png            | 3Z5S2C          | 3Z5S2C          | ✅ MATCH
train-11985.png           | TSQJZ3          | TSQJZ7          | ❌ MISMATCH
train-13577.png           | RCSQ5H  

: 

##  Section 10: Unlabeled Test Data Inference & Final CSV Submission Generation

This final production script processes the completely unlabeled evaluation dataset. It loads the optimized parameters from `best_captcha_crnn.pth`, standardizes the incoming raw test image dimensions via our established single-channel tensor pipeline, and executes forward propagation to yield continuous prediction strings. The outputs are systematically structured and compiled directly into a final `.csv` submission document.

In [ ]:
import os
import glob
import torch
import pandas as pd
from PIL import Image
import torchvision.transforms as transforms
from tqdm import tqdm

# =========================================================
# ⚙️ CONFIGURATION
# =========================================================
TEST_IMAGE_DIR = r"C:\Users\aadit\Downloads\cig_ps\cig_ps\test_images" 
SUBMISSION_CSV_NAME = "submission_Aaditya_Pratap_24113001.csv"

# 1. Verification Checklist
if not os.path.exists(TEST_IMAGE_DIR):
    print(f"⚠️ Error: The directory '{TEST_IMAGE_DIR}' was not found. Please double-check your path string!")
else:
    # 2. Re-instantiate the pristine trained weights
    if os.path.exists("best_captcha_crnn.pth"):
        model.load_state_dict(torch.load("best_captcha_crnn.pth", map_location=device))
        print("💎 Loaded 'best_captcha_crnn.pth' successfully for test inference.")
    else:
        print("⚠️ Checkpoint file not found! Defaulting to the active in-memory model weights.")
        
    model.eval()

    # 3. Use the exact same preprocessing pipeline used during training
    test_transforms = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((64, 200)),  # Matches your training dimensions
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])

    # 4. Gather all test images (.png, .jpg, .jpeg)
    supported_extensions = ["*.png", "*.jpg", "*.jpeg", "*.PNG", "*.JPG"]
    unique_paths = set()  # Use a set to avoid duplicates if any
    for ext in supported_extensions:
        unique_paths.update(glob.glob(os.path.join(TEST_IMAGE_DIR, ext)))
        
    unique_paths = sorted(unique_paths)
    print(f"📋 Found {len(unique_paths)} unlabeled test images to process.")

    # 5. Run Batch-free Inference Loop
    results = []
    
    with torch.no_grad():
        for img_path in tqdm(unique_paths, desc="🔮 Running Model Inference"):
            filename = os.path.basename(img_path)
            try:
                # Open and convert image tensor
                img = Image.open(img_path).convert('L')
                img_tensor = test_transforms(img).unsqueeze(0).to(device) # Add batch dimension
                
                # Execute Forward Pass
                logits = model(img_tensor)
                
                # Decode the model outputs using your existing CTC decoder function
                pred_string = decode_ctc_predictions(logits, num_to_char)[0]
                
                results.append({
                    "image_name": filename,
                    "prediction": pred_string
                })
            except Exception as e:
                print(f"❌ Skipped corrupt file {filename}. Error: {e}")
                results.append({
                    "image_name": filename,
                    "prediction": "ERROR"
                })

    # 6. Save directly to a structured submission CSV file
    df_submission = pd.DataFrame(results)
    df_submission.to_csv(SUBMISSION_CSV_NAME, index=False)
    print(f"\n🎉 Finished! Final submission predictions successfully saved to: {SUBMISSION_CSV_NAME}")
    
    # Quick sanity peek at the first 10 predictions
    print("\n👀 First 10 Sample Predictions:")
    print(df_submission.head(10))